# Stage 7: A Retina-Pretrained Backbone (RETFound) for Diabetic Retinopathy Grading
### Does retina-specific pre-training beat ImageNet pre-training at the same architecture?

**Hypothesis (M3, written before this notebook was run):** [`stage7_hypothesis.md`](stage7_hypothesis.md). Initialising a ViT-Large from the RETFound colour-fundus weights (masked-autoencoder pre-training on retinal photographs) instead of ImageNet raises QWK and the recall of the rare grades.

| Run | Backbone | Pre-training | Role |
|---|---|---|---|
| `retfound_s42/43/44` | ViT-Large/16, 224 px | **RETFound (retina)** | the hypothesis |
| `imagenet_vitl_s42` | ViT-Large/16, 224 px, **same recipe** | ImageNet-21k | control: isolates the pre-training domain |
| Stage 5 arm H (not retrained) | EfficientNet-B3, 300 px | ImageNet | practical comparison |

Everything else is the Stage 5 pipeline: CLAHE + blur, flip/rotation augmentation, Gaussian soft labels (sigma 0.5), cut-offs tuned on validation only. ViT recipe (same for both ViT arms): AdamW, peak lr 3e-4 with layer-wise decay 0.65, weight decay 0.05, 3 warm-up epochs then cosine, drop-path 0.2, 25 epochs, batch 16, gradient clipping 1.0, 5 snapshots.

**Weights.** RETFound is gated on Hugging Face; this notebook uses the ungated `timm`-compatible copy `bitfount/RETFound_MAE` (CFP checkpoint, licence CC-BY-NC-4.0, research use). It is a third-party mirror; the file is loaded with `weights_only=True` and checked against the ImageNet weights in section 4.

## 1. Setup and settings

Every hyper-parameter lives in one `Config`. The defaults are the Stage 3 baseline; an experiment changes only `loss` (`"ce"` or `"mse"`), `seed` and `dedupe`.

In [ ]:
import hashlib
import io
import json
import math
import os
import random
import subprocess
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, fields, replace
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

try:
    import timm
except ImportError:                      # not preinstalled on this machine
    !pip install -q timm
    import timm

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "PDR"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Results and the data cache live on Google Drive, so a Colab disconnect loses nothing.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/dr-grading")      # shared with the Stage 3 notebook (data cache)
    LOCAL_DIR = Path("/content")                               # fast local disk for the unpacked images
except ImportError:                                            # running outside Colab
    WORK_DIR = Path(os.environ.get("DR_WORK_DIR", "dr-grading")).resolve()
    LOCAL_DIR = WORK_DIR
WORK_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU found (very slow!)")

In [ ]:
QUICK_TEST = False        # True = tiny run (few epochs, few batches, separate output folder) to check the pipeline
CHECKPOINT_EVERY = 1      # save a resumable checkpoint every N epochs
SPLIT_SEED = 42           # the train/val/test split is the same for every run (same as Stage 3)


@dataclass
class Config:
    # The Stage 3 baseline `2_aug_lr1e-4`: paper settings plus augmentation and lr 1e-4.
    image_size: int = 224          # ViT-Large/16 has a fixed 224 px input
    epochs: int = 25
    batch_size: int = 16
    lr: float = 3e-4               # peak learning rate of the head; earlier layers get lr * layer_decay^depth
    weight_decay: float = 0.05
    grad_clip: float = 1.0
    n_snapshots: int = 5
    seed: int = 42                # training seed (weights init, batch order, augmentation); the split stays fixed
    backbone: str = "retfound"     # "retfound" (retina weights) | "vit_large_imagenet" (control)
    pretrained: bool = True
    use_amp: bool = True
    augment: bool = True
    loss: str = "soft"            # "ce" = one-hot cross-entropy (Stage 3); "mse" = ordinal regression (Stage 4); "soft" = Gaussian soft labels (Stage 5)
    layer_decay: float = 0.65
    warmup_epochs: int = 3
    drop_path: float = 0.2
    sigma: float = 0.5            # width of the Gaussian soft label, in grades (only for loss="soft")
    dedupe: bool = False          # True = drop conflicting duplicates and keep one copy of each identical image
    max_train_batches: int = 0    # 0 = all batches; >0 only for the quick test


BASE_CFG = Config()
SEEDS = [42, 43, 44]
STAGE7_DIR = WORK_DIR / "stage7"
if QUICK_TEST:
    BASE_CFG = replace(BASE_CFG, epochs=4, n_snapshots=2, warmup_epochs=1, max_train_batches=5)
    SEEDS = [42, 43]
    STAGE7_DIR = WORK_DIR / "stage7_quick_test"            # never mixed with the real runs
if device.type != "cuda" and not QUICK_TEST:
    raise RuntimeError("This runtime has no GPU (Colab falls back to CPU when the free GPU time runs out). "
                       "Runtime -> Change runtime type -> T4 GPU, then Run all again. On a CPU the settings would "
                       "not match the saved runs, and training would take days.")
if device.type != "cuda":
    BASE_CFG = replace(BASE_CFG, use_amp=False)             # mixed precision is a GPU feature
RUNS_DIR, FIG_DIR = STAGE7_DIR / "runs", STAGE7_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {f"retfound_s{seed}": replace(BASE_CFG, backbone="retfound", seed=seed) for seed in SEEDS}
EXPERIMENTS[f"imagenet_vitl_s{SEEDS[0]}"] = replace(BASE_CFG, backbone="vit_large_imagenet", seed=SEEDS[0])      # control (S1)

print(json.dumps(asdict(BASE_CFG), indent=2))
print("experiments:", ", ".join(EXPERIMENTS))

## 2. Data

APTOS 2019: 3,662 labelled fundus photographs, five grades. Same source and cache as Stage 3: the full-resolution Hugging Face mirror [`sngsfydy/aptos`](https://huggingface.co/datasets/sngsfydy/aptos), shrunk to 512 px on the long side, with a zipped copy on Drive. Because the image order comes from the same `labels.csv`, the split below is **identical to Stage 3**.

In [ ]:
HF_REPO = "sngsfydy/aptos"
IMG_DIR = LOCAL_DIR / "aptos512"
DRIVE_CACHE = WORK_DIR / "aptos512.zip"
MAX_SIDE = 512


def list_shards():
    with urllib.request.urlopen(f"https://huggingface.co/api/datasets/{HF_REPO}/tree/main/data") as r:
        return sorted(f["path"] for f in json.load(r) if f["path"].endswith(".parquet"))


def convert_shard(parquet_path, out_dir):
    # Save every image in one parquet shard as a <hash>.jpg (<= 512 px). Returns [(file name, grade), ...].
    rows = []
    pf = pq.ParquetFile(parquet_path)
    for group in range(pf.num_row_groups):
        table = pf.read_row_group(group).to_pydict()
        for img, label in zip(table["image"], table["label"]):
            raw = img["bytes"]
            name = hashlib.sha1(raw).hexdigest()[:16] + ".jpg"     # identical photos get identical names
            im = Image.open(io.BytesIO(raw)).convert("RGB")
            scale = MAX_SIDE / max(im.size)
            im.resize((round(im.width * scale), round(im.height * scale)), Image.LANCZOS).save(out_dir / name, quality=95)
            rows.append((name, int(label)))
    return rows


def download_aptos(out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for shard in list_shards():
        local = LOCAL_DIR / Path(shard).name
        print("downloading", local.name, flush=True)
        subprocess.run(["curl", "-sSL", "--retry", "5", "-o", str(local),
                        f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/{shard}"], check=True)
        rows += convert_shard(local, out_dir)
        local.unlink()                                             # free the disk before the next shard
        print(f"  {len(rows)} images so far", flush=True)
    pd.DataFrame(rows, columns=["id_code", "label"]).to_csv(out_dir / "labels.csv", index=False)


if not (IMG_DIR / "labels.csv").exists():
    if DRIVE_CACHE.exists():                                       # Stage 3 (or an earlier session) did the download
        print("restoring cached images from Drive ...")
        subprocess.run(["unzip", "-q", "-o", str(DRIVE_CACHE), "-d", str(IMG_DIR)], check=True)
    else:
        download_aptos(IMG_DIR)
        print("saving a cache to Drive for next time ...")
        subprocess.run(["zip", "-q", "-r", "-j", str(DRIVE_CACHE), str(IMG_DIR)], check=True)

df = pd.read_csv(IMG_DIR / "labels.csv")
df["path"] = df["id_code"].map(lambda name: IMG_DIR / name)
labels = df["label"].to_numpy()
print(f"{len(df)} images, {df['id_code'].nunique()} distinct files")

### Duplicates and the two splits

APTOS contains byte-identical images, some with **conflicting grades**. The main experiments use the Stage 3 split (all images, stratified 70/15/15, seed 42). The de-duplication check (prediction P8) uses a second split built after **removing every image whose copies disagree** and **keeping one copy of each remaining identical group**, so no test image has a twin in training.

In [ ]:
def file_hash(path):
    return hashlib.md5(path.read_bytes()).hexdigest()


with ThreadPoolExecutor(os.cpu_count()) as pool:
    df["hash"] = list(tqdm(pool.map(file_hash, df["path"]), total=len(df), desc="hashing"))

groups = df.groupby("hash")["label"].agg(["size", "nunique"])
conflict_hashes = set(groups.index[groups["nunique"] > 1])
df["conflicting_duplicate"] = df["hash"].isin(conflict_hashes)
print(f"{int((groups['size'] > 1).sum())} groups of identical images, {len(conflict_hashes)} of them with conflicting grades")


def make_split(positions, seed=SPLIT_SEED, val_frac=0.15, test_frac=0.15):
    # Stratified train/val/test row positions (into df). With all positions this is exactly the Stage 3 split.
    train_val, test = train_test_split(positions, test_size=test_frac, stratify=labels[positions], random_state=seed)
    train, val = train_test_split(train_val, test_size=val_frac / (1 - test_frac),
                                  stratify=labels[train_val], random_state=seed)
    return {"train": np.sort(train), "val": np.sort(val), "test": np.sort(test)}


keep = ~df["conflicting_duplicate"] & ~df.duplicated("hash")      # de-duplicated data set
SPLITS = {False: make_split(np.arange(len(df))), True: make_split(np.flatnonzero(keep.to_numpy()))}

for dedupe, split in SPLITS.items():
    train_hashes = set(df["hash"].to_numpy()[split["train"]])
    leaked = sum(h in train_hashes for h in df["hash"].to_numpy()[split["test"]])
    print(f"\n{'de-duplicated' if dedupe else 'all images (Stage 3 split)'}: {sum(map(len, split.values()))} images, "
          f"{leaked} test images with a twin in training")
    print(pd.DataFrame({name: np.bincount(labels[idx], minlength=NUM_CLASSES) for name, idx in split.items()},
                       index=CLASS_NAMES).T.to_string())

## 3. Pre-processing (unchanged from Stage 3)

CLAHE on the LAB lightness channel (clip 2.0, 8x8 tiles), 5x5 Gaussian blur, square resize to 300 px. Computed once for all images and kept in memory (~1 GB).

In [ ]:
def preprocess(path, size):
    # Load -> shrink to <= 512 px -> CLAHE + blur -> square resize. Returns RGB uint8 (size, size, 3).
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    h, w = bgr.shape[:2]
    if max(h, w) > 512:
        scale = 512 / max(h, w)
        bgr = cv2.resize(bgr, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    bgr = cv2.GaussianBlur(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), (5, 5), 0)
    bgr = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


paths = df["path"].tolist()
images = np.empty((len(paths), BASE_CFG.image_size, BASE_CFG.image_size, 3), dtype=np.uint8)


def fill(i):
    images[i] = preprocess(paths[i], BASE_CFG.image_size)


t0 = time.time()
with ThreadPoolExecutor(os.cpu_count()) as pool:            # OpenCV releases the GIL, so threads are enough
    list(tqdm(pool.map(fill, range(len(paths))), total=len(paths), desc="pre-processing"))
print(f"{images.shape}, {images.nbytes / 1e9:.2f} GB, {time.time() - t0:.0f}s")

## 4. Model: ViT-Large with RETFound or ImageNet weights

The head is one linear layer on the class token (the ViT's own final LayerNorm comes before it). Layer-wise learning-rate decay: the patch embedding and class/position tokens get `lr * decay^24`, block *i* gets `lr * decay^(23-i+1)`... in code, `scale = decay ** (n_layers + 1 - layer_id)`, the head gets scale 1. Biases, norms and the tokens have no weight decay.

**Sanity check S7 (section 4b):** the mirror's patch-embedding weights must differ from the ImageNet ones.

In [ ]:
RETFOUND_URL = "https://huggingface.co/bitfount/RETFound_MAE/resolve/main/pytorch_model.bin"
RETFOUND_FILE = LOCAL_DIR / "retfound_cfp.bin"
VIT_NAME = "vit_large_patch16_224"
IMAGENET_TAG = "vit_large_patch16_224.augreg_in21k_ft_in1k"


def fetch_retfound():
    if not RETFOUND_FILE.exists():
        print("downloading the RETFound CFP weights (1.2 GB) ...")
        subprocess.run(["curl", "-sSL", "--retry", "5", "-o", str(RETFOUND_FILE), RETFOUND_URL], check=True)
    return RETFOUND_FILE


def retfound_state():
    sd = torch.load(fetch_retfound(), map_location="cpu", weights_only=True)      # tensors only: no pickled code is executed
    sd = sd.get("model", sd)
    sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
    return {k: v for k, v in sd.items() if not k.startswith(("head.", "decoder", "mask_token", "fc_norm"))}


def n_outputs(cfg):
    return NUM_CLASSES


class ViTNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        imagenet = cfg.backbone == "vit_large_imagenet"
        self.backbone = timm.create_model(IMAGENET_TAG if imagenet else VIT_NAME, pretrained=imagenet, num_classes=0,
                                          global_pool="token", drop_path_rate=cfg.drop_path)
        if not imagenet:
            missing, unexpected = self.backbone.load_state_dict(retfound_state(), strict=False)
            assert not unexpected, f"unexpected keys in the RETFound file: {unexpected[:5]}"
            assert not missing, f"missing keys: {missing[:5]}"
        self.head = nn.Linear(self.backbone.num_features, NUM_CLASSES)

    def forward(self, x):
        return self.head(self.backbone(x))


def build_model(cfg):
    return ViTNet(cfg).to(device).to(memory_format=torch.channels_last)


def build_optimizer(model, cfg):
    n_layers = len(model.backbone.blocks)
    groups = {}
    for name, p in model.named_parameters():
        if name.startswith("head."):
            layer = n_layers + 1
        elif name.startswith("backbone.blocks."):
            layer = int(name.split(".")[2]) + 1
        elif name.startswith("backbone.norm") or name.startswith("backbone.fc_norm"):
            layer = n_layers + 1
        else:
            layer = 0                                                               # patch embed, cls token, position embedding
        no_decay = p.ndim == 1 or name.endswith(".bias") or "pos_embed" in name or "cls_token" in name
        scale = cfg.layer_decay ** (n_layers + 1 - layer)
        groups.setdefault((layer, no_decay), {"params": [], "lr_scale": scale, "weight_decay": 0.0 if no_decay else cfg.weight_decay})["params"].append(p)
    return torch.optim.AdamW(list(groups.values()), lr=cfg.lr)


def set_epoch_lr(optimizer, cfg, epoch):
    # Linear warm-up for warmup_epochs, then half-cosine to zero. epoch counts from 1.
    if epoch <= cfg.warmup_epochs:
        factor = epoch / cfg.warmup_epochs
    else:
        factor = 0.5 * (1 + math.cos(math.pi * (epoch - cfg.warmup_epochs) / max(cfg.epochs - cfg.warmup_epochs, 1)))
    for g in optimizer.param_groups:
        g["lr"] = cfg.lr * g["lr_scale"] * factor


_mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
_std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)
_black = -_mean / _std


def to_model_input(batch_uint8):
    x = torch.from_numpy(batch_uint8).to(device).permute(0, 3, 1, 2).float() / 255.0
    return ((x - _mean) / _std).contiguous(memory_format=torch.channels_last)


def augment_batch(x):
    # Random horizontal flip + random rotation (0-360 degrees), as in Stages 3-6. Empty corners stay black.
    n = x.shape[0]
    x = x - _black
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)
    angle = torch.rand(n, device=x.device) * 2 * math.pi
    cos, sin, zero = angle.cos(), angle.sin(), torch.zeros_like(angle)
    theta = torch.stack([torch.stack([cos, -sin, zero], 1), torch.stack([sin, cos, zero], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    return (x + _black).contiguous(memory_format=torch.channels_last)

### 4b. Sanity check S7: the mirror holds retina weights, not ImageNet weights

In [ ]:
if True:
    ref = timm.create_model(IMAGENET_TAG, pretrained=True, num_classes=0)
    ret = retfound_state()
    a, b = ret["patch_embed.proj.weight"].float(), ref.patch_embed.proj.weight.detach().float()
    rel = float((a - b).norm() / b.norm())
    same_shape = tuple(a.shape) == tuple(b.shape)
    print(f"patch-embedding weights: same shape {same_shape}, relative difference RETFound vs ImageNet = {rel:.3f}")
    pos_diff = float((ret["pos_embed"].float() - ref.pos_embed.detach().float()).norm() / ref.pos_embed.detach().float().norm())
    print(f"position embedding relative difference = {pos_diff:.3f}")
    (STAGE7_DIR / "mirror_check.json").write_text(json.dumps({"patch_embed_rel_diff": rel, "pos_embed_rel_diff": pos_diff}, indent=2))
    del ref, ret

## 5. From model output to a grade, and the metrics

A model's raw output for an image is either 5 probabilities (`ce`) or one number (`mse`). Both can be turned into a **continuous severity score**: the expected grade Σ k·p(k) for `ce`, the number itself for `mse`. A score becomes a grade through four **cut-offs**: rounding uses 0.5 / 1.5 / 2.5 / 3.5; the tuned rule searches each cut-off on a 0.01 grid to maximise **validation** QWK (coordinate search, three passes). The test set never influences a cut-off.

**Metrics** (all on the test set): QWK, accuracy, macro-F1, recall per grade, errors of two or more grades, the confusion matrix, and the screening view **referable DR = grade >= 2**: sensitivity, specificity and AUC. For a fair AUC across both models, the score for "grade >= k" is Σ_{j>=k} p(j) for `ce` and the regression output for `mse`; *ordinal AUC* is the mean over k = 1..4.

In [ ]:
ROUNDING = np.array([0.5, 1.5, 2.5, 3.5])


def qwk(y, pred, k=NUM_CLASSES):
    # Quadratic weighted kappa from the confusion matrix (same value as sklearn's cohen_kappa_score, much faster).
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k).astype(float)
    weights = np.subtract.outer(np.arange(k), np.arange(k)) ** 2 / (k - 1) ** 2
    expected = np.outer(cm.sum(1), cm.sum(0)) / cm.sum()
    denom = (weights * expected).sum()
    return float(1 - (weights * cm).sum() / denom) if denom > 0 else 0.0


def severity_score(out):
    # (..., 5) probabilities -> expected grade; (..., 1) regression output -> that number.
    return out @ np.arange(NUM_CLASSES) if out.shape[-1] == NUM_CLASSES else out[..., 0]


def at_least_score(out, k):
    # Score for "grade >= k": P(grade >= k) for ce, the regression output for mse (any monotone score works for AUC).
    return out[..., k:].sum(-1) if out.shape[-1] == NUM_CLASSES else out[..., 0]


def apply_cutoffs(score, cutoffs):
    return np.digitize(score, cutoffs).astype(int)


def fit_cutoffs(score, y, step=0.01, passes=3):
    # Coordinate search: move one cut-off at a time to the grid value with the best QWK, keeping them ordered.
    cutoffs = ROUNDING.copy()
    best = qwk(y, apply_cutoffs(score, cutoffs))
    for _ in range(passes):
        for i in range(len(cutoffs)):
            lo = cutoffs[i - 1] + step if i > 0 else -0.5
            hi = cutoffs[i + 1] - step if i < len(cutoffs) - 1 else NUM_CLASSES - 0.5
            for value in np.arange(lo, hi + 1e-9, step):
                trial = cutoffs.copy()
                trial[i] = value
                score_trial = qwk(y, apply_cutoffs(score, trial))
                if score_trial > best:                       # strictly better: ties keep the current cut-off
                    best, cutoffs = score_trial, trial
    return cutoffs


def majority_vote(snapshot_probs):
    # snapshot_probs: (n_snapshots, n_images, 5) -> one predicted grade per image (paper's ensemble rule).
    votes = snapshot_probs.argmax(-1)
    tally = np.stack([(votes == c).sum(0) for c in range(NUM_CLASSES)], 1).astype(float)
    tally += 1e-3 * snapshot_probs.mean(0)                   # tie-break
    return tally.argmax(1)


def safe_auc(y_binary, score):
    return float(roc_auc_score(y_binary, score)) if 0 < y_binary.sum() < len(y_binary) else float("nan")


def compute_metrics(y, pred, out):
    _, rec, _, _ = precision_recall_fscore_support(y, pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    ref_true, ref_pred = y >= 2, pred >= 2
    return {
        "qwk": qwk(y, pred),
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", labels=list(range(NUM_CLASSES)), zero_division=0)),
        **{f"recall_{name}": float(rec[c]) for c, name in enumerate(CLASS_NAMES)},
        "off_by_2_or_more": int(np.sum(np.abs(y - pred) >= 2)),
        "ref_sensitivity": float((ref_pred & ref_true).sum() / max(ref_true.sum(), 1)),
        "ref_specificity": float((~ref_pred & ~ref_true).sum() / max((~ref_true).sum(), 1)),
        "ref_auc": safe_auc(ref_true, at_least_score(out, 2)),
        "ordinal_auc": float(np.nanmean([safe_auc(y >= k, at_least_score(out, k)) for k in range(1, NUM_CLASSES)])),
        "confusion_matrix": confusion_matrix(y, pred, labels=list(range(NUM_CLASSES))).tolist(),
    }

In [ ]:
ARMS = {"ce": ("A", "B"), "mse": ("C", "D"), "soft": ("G", "H")}     # (default rule, tuned-cut-off rule) for each trained model


def decide(run, mode):
    # Grades for the test set from one finished run, for both of its arms.
    # mode "single" = the best-validation epoch; "ensemble" = the 10 snapshots.
    if mode == "single":
        val_out, test_out = run["best_val"], run["best_test"]
    else:
        val_out, test_out = run["snap_val"].mean(0), run["snap_test"].mean(0)
    default_arm, tuned_arm = ARMS[run["config"].loss]
    if run["config"].loss in ("ce", "soft"):       # A / G: arg-max, or the paper's majority vote
        default = test_out.argmax(1) if mode == "single" else majority_vote(run["snap_test"])
    else:                                           # C: round the regression output
        default = apply_cutoffs(severity_score(test_out), ROUNDING)
    cutoffs = fit_cutoffs(severity_score(val_out), run["y_val"])          # B / D: cut-offs from validation only
    tuned = apply_cutoffs(severity_score(test_out), cutoffs)
    return {default_arm: (default, test_out, None), tuned_arm: (tuned, test_out, cutoffs)}

## 5. Training (the Stage 5 loop with a warm-up + cosine schedule)

Best epoch chosen on validation QWK; snapshots every 5th epoch.

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def make_criterion(cfg):
    if cfg.loss == "ce":
        return nn.CrossEntropyLoss()
    if cfg.loss == "soft":                          # Gaussian soft labels: target mass falls off with distance from the true grade
        grades = torch.arange(NUM_CLASSES, device=device).float()
        def soft_ce(out, y):
            target = torch.softmax(-(grades.view(1, -1) - y.float().view(-1, 1)) ** 2 / (2 * cfg.sigma ** 2), dim=1)
            return -(target * F.log_softmax(out, dim=1)).sum(1).mean()
        return soft_ce
    if cfg.loss == "mse":                           # ordinal regression: squared distance to the true grade
        return lambda out, y: F.mse_loss(out[:, 0], y.float())
    raise ValueError(f"unknown loss {cfg.loss!r}")


def default_grades(out):
    # The rule used to pick the best epoch: arg-max for ce, rounding for mse.
    return out.argmax(1) if out.shape[1] == NUM_CLASSES else apply_cutoffs(out[:, 0], ROUNDING)


@torch.no_grad()
def predict(model, indices, cfg, batch_size=32):
    # Softmax probabilities (n, 5) for ce and soft, the raw regression output (n, 1) for mse.
    model.eval()
    out = []
    for start in range(0, len(indices), batch_size):
        x = to_model_input(images[indices[start:start + batch_size]])
        with torch.autocast(device.type, dtype=torch.float16, enabled=cfg.use_amp):
            logits = model(x).float()
        out.append((logits if cfg.loss == "mse" else torch.softmax(logits, dim=1)).cpu())
    return torch.cat(out).numpy()


def train_one_epoch(model, optimizer, scaler, criterion, cfg, split, rng):
    model.train()
    order = rng.permutation(split["train"])
    n_batches = len(order) // cfg.batch_size                                   # drop the last partial batch
    if cfg.max_train_batches:
        n_batches = min(n_batches, cfg.max_train_batches)
    total_loss, seen = 0.0, 0
    progress = tqdm(range(n_batches), desc="  training", leave=False)
    for b in progress:
        idx = np.sort(order[b * cfg.batch_size:(b + 1) * cfg.batch_size])      # sorted = faster memory reads
        x = to_model_input(images[idx])
        if cfg.augment:
            x = augment_batch(x)
        y = torch.from_numpy(labels[idx]).to(device)

        with torch.autocast(device.type, dtype=torch.float16, enabled=cfg.use_amp):
            out = model(x)
        loss = criterion(out.float(), y)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                                             # so clipping sees real gradient sizes
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(y)
        seen += len(y)
        progress.set_postfix(loss=f"{total_loss / seen:.3f}")
    return total_loss / seen

In [ ]:
def save_atomically(path, obj):
    # Write to a temp file, then rename: a crash mid-save can never leave a broken file behind.
    tmp = path.with_suffix(".tmp")
    try:
        torch.save(obj, tmp)
        tmp.replace(path)
    except (RuntimeError, OSError) as err:
        tmp.unlink(missing_ok=True)
        print(f"  warning: could not save {path.name} ({err})")


def load_run(run_dir, cfg):
    saved = np.load(run_dir / "outputs.npz")
    return {"config": cfg, "log": pd.read_csv(run_dir / "log.csv"), **{k: saved[k] for k in saved.files}}


def run_experiment(name, cfg):
    # Train one configuration and save its validation/test outputs. Resumes if interrupted.
    run_dir = RUNS_DIR / name
    run_dir.mkdir(parents=True, exist_ok=True)
    config_file, done_file = run_dir / "config.json", run_dir / "results.json"
    ckpt_path, best_path = run_dir / "last.pt", run_dir / "best_val_weights.pt"

    saved_cfg = json.loads(config_file.read_text()) if config_file.exists() else None
    if saved_cfg is not None:
        saved_cfg = {**{f.name: f.default for f in fields(Config)}, **saved_cfg}
    if saved_cfg is not None and saved_cfg != asdict(cfg) and done_file.exists():
        # Never throw away a finished run because a setting differs (e.g. a CPU runtime turns AMP off).
        changed = {k: (saved_cfg.get(k), v) for k, v in asdict(cfg).items() if saved_cfg.get(k) != v}
        raise RuntimeError(f"{name} already finished with other settings {changed} (saved, now). "
                           f"Delete {run_dir} by hand if you really want to retrain it.")
    if saved_cfg != asdict(cfg):              # a new experiment, or an unfinished one whose settings changed
        ckpt_path.unlink(missing_ok=True)
        done_file.unlink(missing_ok=True)
        config_file.write_text(json.dumps(asdict(cfg), indent=2))
    if done_file.exists():                    # finished in an earlier session
        print(f"=== {name} === already finished, loading saved outputs")
        return load_run(run_dir, cfg)

    split = SPLITS[cfg.dedupe]
    print(f"\n=== {name} ===  loss={cfg.loss}  backbone={cfg.backbone}  sigma={cfg.sigma}  seed={cfg.seed}  lr={cfg.lr}  epochs={cfg.epochs}")
    seed_everything(cfg.seed)
    model = build_model(cfg)
    optimizer = build_optimizer(model, cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    criterion = make_criterion(cfg)
    rng = np.random.default_rng(cfg.seed)

    snapshot_epochs = {round(cfg.epochs * k / cfg.n_snapshots) for k in range(1, cfg.n_snapshots + 1)}
    progress = {"log": [], "snap_val": [], "snap_test": [], "best_val_qwk": -2.0, "best_epoch": 0,
                "best_val": None, "best_test": None}
    first_epoch = 1

    if ckpt_path.exists():                                                     # continue an interrupted run
        try:
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        except Exception as err:
            ckpt = None
            print(f"  could not read {ckpt_path.name} ({err}); starting from scratch")
        if ckpt is not None:
            model.load_state_dict(ckpt["model"])
            optimizer.load_state_dict(ckpt["optimizer"])
            scaler.load_state_dict(ckpt["scaler"])
            rng.bit_generator.state = ckpt["rng_numpy"]
            torch.set_rng_state(ckpt["rng_torch"])
            if ckpt["rng_cuda"] is not None:
                torch.cuda.set_rng_state_all(ckpt["rng_cuda"])
            progress, first_epoch = ckpt["progress"], ckpt["epoch"] + 1
            print(f"  RESUMED after epoch {ckpt['epoch']} (best so far: val QWK {progress['best_val_qwk']:.3f})")

    y_val = labels[split["val"]]
    started, epochs_done_now = time.time(), 0
    for epoch in range(first_epoch, cfg.epochs + 1):
        t0 = time.time()
        set_epoch_lr(optimizer, cfg, epoch)
        train_loss = train_one_epoch(model, optimizer, scaler, criterion, cfg, split, rng)

        val_out = predict(model, split["val"], cfg)
        val_pred = default_grades(val_out)
        val_qwk, val_acc = qwk(y_val, val_pred), float(accuracy_score(y_val, val_pred))
        progress["log"].append({"epoch": epoch, "train_loss": train_loss, "val_qwk": val_qwk, "val_acc": val_acc})

        notes, test_out = "", None
        if val_qwk > progress["best_val_qwk"]:                                 # new best on validation
            test_out = predict(model, split["test"], cfg)
            progress.update(best_val_qwk=val_qwk, best_epoch=epoch, best_val=val_out, best_test=test_out)
            save_atomically(best_path, model.state_dict())
            notes += " NEW BEST"
        if epoch in snapshot_epochs:                                           # ensemble member
            progress["snap_val"].append(val_out)
            progress["snap_test"].append(test_out if test_out is not None else predict(model, split["test"], cfg))
            notes += f" snapshot {len(progress['snap_test'])}/{cfg.n_snapshots}"

        pd.DataFrame(progress["log"]).to_csv(run_dir / "log.csv", index=False)    # live log, readable at any time
        if epoch % CHECKPOINT_EVERY == 0 and epoch < cfg.epochs:
            save_atomically(ckpt_path, {
                "epoch": epoch, "progress": progress, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(), "rng_numpy": rng.bit_generator.state, "rng_torch": torch.get_rng_state(),
                "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None})

        epochs_done_now += 1
        eta_min = (time.time() - started) / epochs_done_now * (cfg.epochs - epoch) / 60
        print(f"[{name}] epoch {epoch:2d}/{cfg.epochs} | loss {train_loss:.3f} | val QWK {val_qwk:.3f} "
              f"(best {progress['best_val_qwk']:.3f} @ep{progress['best_epoch']}) | val acc {val_acc:.3f} | "
              f"{time.time() - t0:.0f}s | ~{eta_min:.0f} min left |{notes}")

    np.savez_compressed(run_dir / "outputs.npz", y_val=y_val, y_test=labels[split["test"]],
                        val_idx=split["val"], test_idx=split["test"],
                        best_val=progress["best_val"], best_test=progress["best_test"],
                        snap_val=np.stack(progress["snap_val"]), snap_test=np.stack(progress["snap_test"]))
    done_file.write_text(json.dumps({"best_epoch": progress["best_epoch"],              # written last = run complete
                                     "best_val_qwk": progress["best_val_qwk"]}, indent=2))
    ckpt_path.unlink(missing_ok=True)
    return load_run(run_dir, cfg)

## 7. Run the experiments

Order: `retfound_s42`, `imagenet_vitl_s42` (the control, so S1 can be read early), `retfound_s43`, `retfound_s44`.

In [ ]:
ORDER = [f"retfound_s{SEEDS[0]}", f"imagenet_vitl_s{SEEDS[0]}"] + [f"retfound_s{s}" for s in SEEDS[1:]]
runs = {}
for name in ORDER:
    runs[name] = run_experiment(name, EXPERIMENTS[name])

## 8. Baseline: the Stage 5 EfficientNet-B3 soft-label runs (arm H)

Read from the output of the Kaggle kernel `dr-s5-train` (attached as an input). Same split and test set.

In [ ]:
INPUT_DIR = Path("/kaggle/input")
base_runs = {}
for f in sorted(INPUT_DIR.glob("**/dr-grading/stage5/runs/soft_s4*/outputs.npz")):
    saved = np.load(f)
    seed = int(f.parent.name.rsplit("_s", 1)[1])
    base_runs[seed] = {"config": replace(BASE_CFG, backbone="efficientnet_b3", loss="soft", seed=seed), **{k: saved[k] for k in saved.files}}
print("Stage 5 baseline runs found for seeds:", sorted(base_runs))
if not base_runs:
    raise RuntimeError("Attach the kernel dr-s5-train as an input: its soft_s42/43/44 outputs are the baseline.")
common = [s for s in SEEDS if s in base_runs]
assert all(np.array_equal(base_runs[s]["y_test"], runs[f"retfound_s{s}"]["y_test"]) for s in common), "test sets differ"

## 9. Results: every arm, every seed

ViT runs use the soft-label loss, so they have the same two arms: **G** (vote over the 5 snapshots) and **H** (expected grade, validation-tuned cut-offs).

In [ ]:
class _Cfg:                                    # decide() only reads .loss; the ViT runs are trained with the soft-label loss
    loss = "soft"

SHOW = ["qwk", "accuracy", "macro_f1", "recall_Mild", "recall_Severe", "recall_PDR", "off_by_2_or_more", "ref_auc"]
all_runs = {("EffNet-B3 ImageNet (S5)", f"soft_s{s}"): base_runs[s] for s in common}
for name, run in runs.items():
    all_runs[("ViT-L RETFound" if name.startswith("retfound") else "ViT-L ImageNet", name)] = run

rows, decisions = [], {}
for (group, name), run in all_runs.items():
    run = {**run, "config": _Cfg()}
    for mode in ("single", "ensemble"):
        for arm, (pred, out_, cutoffs) in decide(run, mode).items():
            y = run["y_test"]
            decisions[(group, name, mode, arm)] = (y, pred, out_)
            m = compute_metrics(y, pred, out_)
            rows.append({"group": group, "run": name, "mode": mode, "arm": arm, **{k: v for k, v in m.items() if k != "confusion_matrix"}})
res = pd.DataFrame(rows)
res.to_csv(STAGE7_DIR / "results_per_run.csv", index=False)
agg = res.groupby(["group", "mode", "arm"])[SHOW].agg(["mean", "std"])
table = pd.DataFrame({c: agg[(c, "mean")].map("{:.3f}".format) + " ± " + agg[(c, "std")].fillna(0).map("{:.3f}".format) for c in SHOW})
table.to_csv(STAGE7_DIR / "results_mean_over_seeds.csv")
display(table)
display(res[res["mode"] == "ensemble"].style.format(precision=3).hide(axis="index"))

### Paired bootstrap (seed 42, same 550 test images)

In [ ]:
def paired_bootstrap(y, pred_a, pred_b, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[b] = qwk(y[idx], pred_b[idx]) - qwk(y[idx], pred_a[idx])
    return qwk(y, pred_b) - qwk(y, pred_a), *np.percentile(diffs, [2.5, 97.5])

s0 = SEEDS[0]
get = lambda group, name, arm: decisions[(group, name, "ensemble", arm)]
y0 = get("ViT-L RETFound", f"retfound_s{s0}", "H")[0]
pairs = [("RETFound - ImageNet ViT-L (tuned)", get("ViT-L ImageNet", f"imagenet_vitl_s{s0}", "H")[1], get("ViT-L RETFound", f"retfound_s{s0}", "H")[1]),
         ("RETFound - ImageNet ViT-L (vote)", get("ViT-L ImageNet", f"imagenet_vitl_s{s0}", "G")[1], get("ViT-L RETFound", f"retfound_s{s0}", "G")[1]),
         ("RETFound ViT-L - EffNet-B3 S5 (tuned)", get("EffNet-B3 ImageNet (S5)", f"soft_s{s0}", "H")[1], get("ViT-L RETFound", f"retfound_s{s0}", "H")[1])]
boot = []
for label, pa, pb in pairs:
    d, lo, hi = paired_bootstrap(y0, pa, pb)
    boot.append({"contrast": label, "delta_qwk": d, "ci_low": lo, "ci_high": hi, "excludes_0": lo > 0 or hi < 0})
bootstrap = pd.DataFrame(boot)
bootstrap.to_csv(STAGE7_DIR / "paired_bootstrap.csv", index=False)
display(bootstrap.style.format(precision=3).hide(axis="index"))

## 10. Were the predictions right? (M4)

Thresholds copied from [`stage7_hypothesis.md`](stage7_hypothesis.md), section 4, fixed before the first run.

In [ ]:
PREDICTIONS = {
    "S1_min_qwk_gain": 0.01,          # RETFound - ImageNet ViT-L, tuned, seed 42
    "S2_min_rare_recall_gain": 0.05,  # mean of Severe and PDR recall
    "S3_max_abs_diff_vs_cnn": 0.015,  # abs(R - H) in QWK, mean over seeds
    "S4_max_off2_factor": 1.10,
    "S5_min_vit_std": 0.005,          # std over seeds of RETFound tuned QWK > this
    "S6_max_auc_drop": 0.005,
    "S7_min_rel_diff": 1e-3,
}
num = lambda s: float(str(s).split()[0])
ens = table.xs("ensemble", level="mode")
v = lambda group, arm, col: num(ens.loc[(group, arm), col])
R, I, B3 = "ViT-L RETFound", "ViT-L ImageNet", "EffNet-B3 ImageNet (S5)"
rr = res[(res["mode"] == "ensemble") & (res["arm"] == "H")]
one = lambda grp, col, name_suffix=f"_s{SEEDS[0]}": float(rr[(rr["group"] == grp) & (rr["run"].str.endswith(name_suffix))][col].iloc[0])

checks = []
d1 = one(R, "qwk") - one(I, "qwk")
checks.append(("S1", "RETFound beats ImageNet ViT-L by >= +0.01 QWK (tuned, seed 42)", f"{one(R,'qwk'):.3f} vs {one(I,'qwk'):.3f} ({d1:+.3f})", d1 >= PREDICTIONS["S1_min_qwk_gain"]))
rare = lambda grp: (one(grp, "recall_Severe") + one(grp, "recall_PDR")) / 2
d2 = rare(R) - rare(I)
checks.append(("S2", "mean(Severe, PDR) recall gain >= +0.05 (seed 42)", f"{rare(I):.3f} -> {rare(R):.3f} ({d2:+.3f})", d2 >= PREDICTIONS["S2_min_rare_recall_gain"]))
d3 = v(R, "H", "qwk") - v(B3, "H", "qwk")
checks.append(("S3", "RETFound within 0.015 QWK of the EfficientNet-B3 (mean over seeds)", f"{v(R,'H','qwk'):.3f} vs {v(B3,'H','qwk'):.3f} ({d3:+.3f})", abs(d3) < PREDICTIONS["S3_max_abs_diff_vs_cnn"]))
checks.append(("S4", "errors >= 2 grades <= 1.10 x the CNN", f"{v(B3,'H','off_by_2_or_more'):.1f} -> {v(R,'H','off_by_2_or_more'):.1f}", v(R, "H", "off_by_2_or_more") <= PREDICTIONS["S4_max_off2_factor"] * v(B3, "H", "off_by_2_or_more")))
std_r = float(rr[rr["group"] == R]["qwk"].std())
checks.append(("S5", "std over seeds of RETFound tuned QWK > 0.005", f"std {std_r:.4f} over {int((rr['group'] == R).sum())} seeds", std_r > PREDICTIONS["S5_min_vit_std"]))
checks.append(("S6", "referable AUC >= CNN - 0.005", f"{v(B3,'H','ref_auc'):.3f} -> {v(R,'H','ref_auc'):.3f}", v(R, "H", "ref_auc") >= v(B3, "H", "ref_auc") - PREDICTIONS["S6_max_auc_drop"]))
mirror = json.loads((STAGE7_DIR / "mirror_check.json").read_text())
checks.append(("S7", "sanity: mirror weights differ from ImageNet (patch embedding)", f"relative difference {mirror['patch_embed_rel_diff']:.3f}", mirror["patch_embed_rel_diff"] > PREDICTIONS["S7_min_rel_diff"]))
verdict = pd.DataFrame(checks, columns=["id", "prediction", "observed", "confirmed"])
verdict.to_csv(STAGE7_DIR / "prediction_check.csv", index=False)
print(f"{int(verdict['confirmed'].sum())} of {len(verdict)} predictions confirmed")
verdict.style.hide(axis="index")

## 11. Figures and per-grade recall

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (title, key) in zip(axes, [("EffNet-B3 ImageNet (S5), H", (B3, f"soft_s{s0}")), ("ViT-L ImageNet, H", (I, f"imagenet_vitl_s{s0}")), ("ViT-L RETFound, H", (R, f"retfound_s{s0}"))]):
    y, pred, _ = decisions[(key[0], key[1], "ensemble", "H")]
    cm = confusion_matrix(y, pred, labels=list(range(NUM_CLASSES)))
    ax.imshow(cm, cmap="Blues")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30); ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_title(f"{title}\nQWK {qwk(y, pred):.3f}"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.tight_layout(); fig.savefig(FIG_DIR / "confusion_matrices.png", dpi=150); plt.show()

recall = lambda y, pred: pd.Series([(pred[y == c] == c).mean() for c in range(NUM_CLASSES)], index=CLASS_NAMES)
per_class = pd.DataFrame({label: recall(*decisions[(g, n, "ensemble", "H")][:2]) for label, (g, n) in
                          {"EffNet-B3 (S5)": (B3, f"soft_s{s0}"), "ViT-L ImageNet": (I, f"imagenet_vitl_s{s0}"), "ViT-L RETFound": (R, f"retfound_s{s0}")}.items()})
per_class["test images"] = np.bincount(y0, minlength=NUM_CLASSES)
per_class.to_csv(STAGE7_DIR / "per_class_recall.csv")
display(per_class.style.format(precision=3))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, name in zip(axes, [f"retfound_s{s0}", f"imagenet_vitl_s{s0}"]):
    log = runs[name]["log"]
    ax.plot(log["epoch"], log["val_qwk"], label="val QWK"); ax.plot(log["epoch"], log["train_loss"], label="train loss")
    ax.set_title(name); ax.set_xlabel("epoch"); ax.set_ylim(0, 1); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "curves.png", dpi=150); plt.show()

## 12. Files

Written to `dr-grading/stage7/` in this kernel's output: `results_per_run.csv`, `results_mean_over_seeds.csv`, `paired_bootstrap.csv`, `prediction_check.csv`, `per_class_recall.csv`, `mirror_check.json`, `figures/`, `runs/<name>/`. Copy the CSVs and figures into `dr-repro/results/stage7/` and write `stage7_results.md`.